# F6-svd-spectral — Practice p25 — Solution

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
W_basis = rng.integers(-5, 6, size=(9, 3))
W = np.column_stack(
    (W_basis, W_basis[:, 0] + W_basis[:, 1] - W_basis[:, 2])
)

# (a) A 9-by-4 factor forces at least 9 - 4 zero eigenvalues.
predicted_zero_count = W.shape[0] - W.shape[1]

# Each pair of rows is multiplied coordinatewise, then reduced once.
S = (W[:, None, :] * W[None, :, :]).sum(axis=2)
symmetry_ok = bool(np.allclose(S, S.T, atol=1e-12, rtol=0))

In [ ]:
# (b) eigh is ascending, so the same reversal acts on values and columns.
assert symmetry_ok
lam_asc, Q_asc = np.linalg.eigh(S)
lam_desc = lam_asc[::-1]
Q_desc = Q_asc[:, ::-1]
S_reconstructed = (Q_desc * lam_desc) @ Q_desc.T
reconstruction_gap = np.linalg.norm(S_reconstructed - S, ord="fro")
assert np.isclose(reconstruction_gap, 0.0, atol=1e-9, rtol=0)

In [ ]:
# (c) Count the numerical zero block, then infer the revealed rank.
assert np.isclose(reconstruction_gap, 0.0, atol=1e-9, rtol=0)
observed_zero_count = int(
    np.isclose(lam_desc, 0.0, atol=1e-9, rtol=0).sum()
)
rank_from_spectrum = lam_desc.size - observed_zero_count
assert observed_zero_count >= predicted_zero_count

The shape-only prediction was five zero eigenvalues, but the spectrum shows six. Shape gives only $\operatorname{rank}(W)\leq4$; here the last column of `W` is an exact linear combination of the first three, so the actual rank is three and $9-3=6$ eigenvalues of $S$ are zero.

In [ ]:
# (d) Compare spectra, not eigenvectors inside the degenerate zero block.
assert observed_zero_count >= predicted_zero_count
sigma = np.linalg.svd(W, compute_uv=False)
bridge_gap = np.max(np.abs(lam_desc[:4] - sigma**2))
zero_block_energy = np.sum(lam_desc[rank_from_spectrum:]**2)
assert np.isclose(bridge_gap, 0.0, atol=1e-9, rtol=0)
assert np.isclose(zero_block_energy, 0.0, atol=1e-9, rtol=0)

In [ ]:
# (e) One reverse cumulative pass contains every requested tail sum.
assert np.isclose(bridge_gap, 0.0, atol=1e-9, rtol=0)
assert np.isclose(zero_block_energy, 0.0, atol=1e-9, rtol=0)
spectrum_for_tail = np.where(
    np.isclose(lam_desc, 0.0, atol=1e-9, rtol=0), 0.0, lam_desc
)
tail_sq = np.cumsum((spectrum_for_tail**2)[::-1])[::-1]
rel_err2 = tail_sq[1:5] / tail_sq[0]
rank3_rel_err2 = rel_err2[2]
rank4_rel_err2 = rel_err2[3]
rank4_buys_nothing = bool(
    np.isclose(rank3_rel_err2, rank4_rel_err2, atol=1e-12, rtol=0)
)
assert np.isclose(rank3_rel_err2, 0.0, atol=1e-12, rtol=0)
assert np.isclose(rank4_rel_err2, 0.0, atol=1e-12, rtol=0)
assert rank4_buys_nothing

print("W:\n", W)
print("eigenvalues (descending):", lam_desc)
print("singular values squared:", sigma**2)
print("relative squared errors r=1..4:", rel_err2)
print("predicted / observed zeros:", predicted_zero_count, observed_zero_count)

The shape of `W` guaranteed only five zero eigenvalues, whereas `observed_zero_count = 6` reveals `rank_from_spectrum = 3`. Those six zero-eigenvalue directions form the null space of $W^{\mathsf T}$: six independent coefficient directions among the nine rows collapse to zero. Since `rank3_rel_err2` and `rank4_rel_err2` are both zero and `rank4_buys_nothing` is true, the rank-3 truncation already equals $S$ and a fourth direction adds nothing. Thus the useful ceiling is the revealed rank, not the column count and not a threshold selected from an error budget.

### Answer check

In [ ]:
expected_positive_lam = np.array([
    283.47785415610633,
    90.40164926854435,
    74.12049657534930,
])
expected_rel_err2 = np.array([
    0.14534603410811134,
    0.058429030401977854,
    0.0,
    0.0,
])

assert W.shape == (9, 4) and S.shape == (9, 9)
assert symmetry_ok is True
assert lam_desc.shape == (9,) and Q_desc.shape == (9, 9)
np.testing.assert_allclose(
    lam_desc[:3], expected_positive_lam, atol=1e-10, rtol=0
)
np.testing.assert_allclose(
    lam_desc[3:], np.zeros(6), atol=1e-9, rtol=0
)
assert np.isclose(reconstruction_gap, 0.0, atol=1e-9, rtol=0)
assert np.isclose(bridge_gap, 0.0, atol=1e-9, rtol=0)
assert np.isclose(zero_block_energy, 0.0, atol=1e-9, rtol=0)
np.testing.assert_allclose(rel_err2, expected_rel_err2, atol=1e-12, rtol=0)
assert predicted_zero_count == 5
assert observed_zero_count == 6
assert rank_from_spectrum == 3
assert rank4_buys_nothing is True